# 5주차 2차시: 스코어 필터링 · 문서 압축 · 평가 지표

| 주제 | 내용 |
|---|---|
| 스코어 필터링 | Fixed / Dynamic / Score Gap · AdaptiveFilter |
| 문서 압축 | Extractive · LLM · MMR |
| 평가 지표 | AP · mAP · ILS |

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
# from dotenv import load_dotenv
# load_dotenv()
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

## 스코어 필터링 (Score Filtering)

1차시 Reranking 결과에서 관련성이 낮은 문서를 제거해 컨텍스트 품질을 높임.

| 전략 | 방식 | 특징 |
|---|---|---|
| Fixed Threshold | 고정 점수 이상만 유지 | 단순, 도메인별 임계값 설정 필요 |
| Dynamic Threshold | 평균 - 표준편차 기준 | 데이터 분포에 적응적 |
| Score Gap | 점수 급락 구간에서 절단 | 자연스러운 경계 탐지 |

In [ ]:
class ScoreFilter:
  # self 안적어도 사용할 수 있는 방법이 있음.

  @staticmethod
  def fixed_threshold(scored_docs, threshold = 0.5):
    return [(doc, s) for doc, s in scored_docs if s >= threshold]

  @staticmethod
  def dynamic_threshold(scored_docs, std_factor):  # mean - ???
    scores = [s for _, s in scored_docs]
    threshold = np.mean(scores) - std_factor * np.std(scores)
    return [(doc, s) for doc, s in scored_docs if s >= threshold]

  @staticmethod
  def score_gap(scored_docs, min_docs=2):
    if len(scored_docs) <= min_docs:
      return scored_docs

    scores = [s for _, s in scored_docs]
    gaps = [(scores[i] - scores[i+1], i+1) for i in range(len(scores)-1)]
    max_gap_idx = max(gaps, key = lambda x:x[0])[1]
    cut = max(min_docs, max_gap_idx)
    return scored_docs[:cut]

In [ ]:
test_scores = [(Document(page_content="", metadata={'id': f'd{i}'}), s) for i, s in enumerate([0.9, 0.85, 0.7, 0.5, 0.2])]
display(test_scores)

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5),
 (Document(metadata={'id': 'd4'}, page_content=''), 0.2)]

#### Fixed Threshold

점수가 threshold(0.5) 이상인 문서만 반환.

In [ ]:
# 굳이 메서드들이 초기화가 필요없는 그냥 단순 메서드(헬퍼 등)라면
# 그냥 객체를 만들지 않고도 클래스 이름으로 쓰면 됨
  # 근데 @staticmethod 등을 안써도 작동은 함! -> 의도 명확화 & 객체로 호출해도 동작하도록 하는 목적
filtered = ScoreFilter.fixed_threshold(test_scores, threshold = 0.5)

filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

#### Dynamic Threshold

`평균 - std_factor × 표준편차` 이상인 문서만 반환.

In [ ]:
filtered = ScoreFilter.dynamic_threshold(test_scores, std_factor = 1)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

#### Score Gap

연속된 문서 간 점수 차이가 가장 큰 지점에서 절단.

In [ ]:
filtered = ScoreFilter.score_gap(test_scores, min_docs = 2)
filtered

[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

### 적응형 필터링 (AdaptiveFilter)

문서 점수의 표준편차와 범위를 분석해 최적 전략 자동 선택.

- `std > 0.2` → Score Gap 사용
- `score_range < 0.1` → Top Half (상위 50%) 반환
- 그 외 → Dynamic Threshold 사용

In [ ]:
class AdaptiveFilter:

    @staticmethod
    def apply(scored_docs):
        scores = [s for _, s in scored_docs]
        print(f"scores: {scores}")
        std = np.std(scores) # 문서들 점수의 std를 구하고 조건문에 활용
        score_range = max(scores) - min(scores) # 문서들 점수의 범위를 구함

        if std > 0.2:
            print("Score Gap")
            return ScoreFilter.score_gap(scored_docs)

        elif score_range < 0.1:
            print("Top Half")
            cut_idx = max(1, len(scored_docs) // 2)
            return scored_docs[:cut_idx]

        else:
            print("Threshold")
            return ScoreFilter.dynamic_threshold(scored_docs, std_factor=1.0)

In [ ]:
AdaptiveFilter.apply(test_scores)

scores: [0.9, 0.85, 0.7, 0.5, 0.2]
Score Gap


[(Document(metadata={'id': 'd0'}, page_content=''), 0.9),
 (Document(metadata={'id': 'd1'}, page_content=''), 0.85),
 (Document(metadata={'id': 'd2'}, page_content=''), 0.7),
 (Document(metadata={'id': 'd3'}, page_content=''), 0.5)]

## 문서 압축 (Context Compression)

검색 문서 전체를 LLM에 넘기면 토큰 비용 증가 + 노이즈 증가.  
질문 관련 핵심 내용만 추출해 컨텍스트 품질을 높이는 과정.

| 방식 | 설명 | 특징 |
|---|---|---|
| Extractive | 쿼리 키워드 겹침 기반 문장 선택 | 빠름, 단순 |
| LLM 기반 | LLM이 관련 문장만 추출/생성 | 높은 정확도, API 비용 |
| MMR | 관련성 + 다양성 균형 최적화 | 중복 제거 효과적 |

### Extractive Compression

쿼리 용어와의 겹침(overlap) 점수로 문장 선택. 단순하고 빠름.

In [ ]:
import re
def extractive_compress(query, document, max_sentences=3):
#   sentences = document.split('.')
  sentences = re.split(r'[.!?]\s*', document)

  if not sentences:
    return document

  query_terms = set(query.lower().split())

  scored = []
  for sent in sentences:
    sent_terms = set(sent.lower().split())
    overlap = len(query_terms & sent_terms)
    scored.append((sent, overlap))

  scored.sort(key = lambda x: x[1], reverse = True)
  selected = [s for s, _ in scored[:max_sentences]]

  return ". " .join(selected) + "."

In [ ]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"
compressed = extractive_compress(query, long_doc, max_sentences = 3)

In [ ]:
compressed

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

### LLM 기반 압축

압축 프롬프트를 정의하고 질문 관련 문장만 추출하도록 LLM에 요청.

In [ ]:
compress_chain = ChatPromptTemplate.from_messages([
    ('system', "당신은 문서 요약 전문가입니다."),
    ('human', """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요

    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.

    쿼리 : {query}
    문서 : {document}


    압축결과: """)

]) | llm | StrOutputParser()

In [ ]:
def llm_compress(query, document, max_tokens=100):
  return compress_chain.invoke({
      'query' : query,
      'document' : document,
      'max_tokens' : max_tokens

  })

In [ ]:
compressed_llm = llm_compress(query, long_doc)

In [ ]:
compressed_llm

'트랜스포머는 2017년 구글 발표의 아키텍처로, BERT와 GPT의 기반입니다.'

### Contextual Compression Retriever

`LLMChainExtractor` + `ContextualCompressionRetriever`를 결합해 검색과 압축을 하나의 파이프라인으로 처리.

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]
vectorstore = FAISS.from_documents(documents, embeddings_model)

In [ ]:
compressor_retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = vectorstore.as_retriever(search_kwargs={'k':3}))

compressed_docs = compressor_retriever.invoke(query)

In [ ]:
# 이건 LangChain의 내장 기능을 사용하여, 검색된 여러 문서들(Document 객체) 중에서
  # 쿼리와 관련이 적은 내용을 삭제하거나 핵심 문구만 추출하여 Document 객체 리스트로 유지
compressed_docs

# 벡터스토어에서 쿼리를 이용해 뽑아온 문서를, compressor로 압축해서 2개를 내보냄

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

### 압축 품질 평가

- **압축률**: 원본 대비 압축 후 길이 비율
- **키워드 보존율**: 쿼리 핵심 단어가 얼마나 남았는지
- **의미 유사도**: 원본과 압축본의 임베딩 코사인 유사도

In [ ]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
def get_embedding(text):
  return np.array(embeddings_model.embed_query(text))

In [ ]:
def evaluate_compression(original, compressed, query):
  # 1) 압축률
  ratio = len(compressed) / len(original)

  # 2) 키워드 보존율
  query_terms = set(query.lower().split())
  orig_terms = set(original.lower().split())
  comp_terms = set(compressed.lower().split())

  orig_query_terms = query_terms & orig_terms
  comp_query_terms = query_terms & comp_terms
  keyword_coverage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0

  # 3) 의미 보존도
  original_emb = np.array(embeddings_model.embed_query(original))
  compressed_emb = get_embedding(compressed)
  cosine_similarity = np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

  semantic_similarity = cosine_similarity

  return {
      'compression_ratio' : ratio,
      'keyword_coverage' : keyword_coverage,
      'semantic_similarity' : semantic_similarity
  }


In [ ]:
evaluate_compression(long_doc, compressed_llm, query)

{'compression_ratio': 0.29931972789115646,
 'keyword_coverage': 1.0,
 'semantic_similarity': np.float64(0.7912176990811298)}

### MMR 기반 압축 (Maximum Marginal Relevance)

관련성(Relevance)과 다양성(Diversity)을 동시에 최적화.

```
MMR(s) = λ × Relevance(s, query) - (1-λ) × max Similarity(s, selected)
```

- λ 높음 → 관련성 우선 (정보량 집중)
- λ 낮음 → 다양성 우선 (중복 제거)

In [ ]:
def cosine_sim(original_emb, compressed_emb):
  return np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

In [ ]:
def mmr_compress(query, document, max_sentences = 3, param = 0.5):
  sentences = re.split(r'[.!?]\s*', document)
  sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

  if len(sentences) <= max_sentences:
    return ". ".join(sentences) + "."

  query_emb = get_embedding(query)
  sent_embs = [get_embedding(s) for s in sentences]

  selected = []
  remaining = list(range(len(sentences)))

  for _ in range(max_sentences):
    best_score = -float('inf') # 가장 작은 값
    best_idx = -1

    for idx in remaining:
      relevance = cosine_sim(query_emb, sent_embs[idx])

      if selected:
        max_sim = max(cosine_sim(sent_embs[idx], sent_embs[s]) for s in selected)
      else:
        max_sim = 0.0

      mmr = param * relevance - (1 - param) * max_sim

      if mmr > best_score:
        best_score = mmr
        best_idx = idx

    selected.append(best_idx)
    remaining.remove(best_idx)

  return ". ".join(sentences[i] for i in sorted(selected)) + "."




#### Relevance 중심 (param=0.7)

관련성 70% 반영. 쿼리와 직접 관련된 문장 위주로 선택.

In [ ]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.'

#### Diversity 중심 (param=0.3)

다양성 70% 반영. 중복 없이 다양한 관점의 문장 선택.

In [ ]:
mmr_compress(query, long_doc, max_sentences=2, param=0.3)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

## 검색 품질 평가 지표

재순위화 · 필터링이 실제로 검색 품질을 개선했는지 정량 측정.

### AP (Average Precision)

각 관련 문서 위치에서의 Precision 평균. 1에 가까울수록 관련 문서가 상위에 밀집.

```
예: [관련O, 관련X, 관련O, 관련X, 관련O]
P@k:  1/1     -     2/3     -     3/5
AP = (1/1 + 2/3 + 3/5) / 3 = 0.756
```

In [ ]:
def average_precision(retrieved_ids, relevant_ids):
  relevant_set = set(relevant_ids)
  hits = 0
  sum_precision = 0.0

  for i, doc_id in enumerate(retrieved_ids):
    if doc_id in relevant_set:
      hits +=1
      precision_at_i = hits / (i+1)
      sum_precision += precision_at_i

  return sum_precision/len(relevant_set) if relevant_set else 0.0


### mAP (Mean Average Precision)

여러 질문에 대한 AP의 평균. 시스템 전체 성능을 하나의 수치로 표현.

In [ ]:
def mean_average_precision(query_results):
  # query_results = {query : {retrieved_ids, relevant_ids}, query2:{retrieved_ids, relevant_ids}}

  aps = []
  for query, (retrieved, relevant) in query_results.items():
    ap = average_precision(retrieved, relevant)
    aps.append(ap)

  # aps = [0.7556, 0.7777, 0.3333, ...]
  map_score = np.mean(aps)
  return map_score

### 실행: 벡터 검색 결과에 AP 적용

In [ ]:
def vector_search(query, vectorstore, top_k = 5):
  results = vectorstore.similarity_search_with_score(query, k= top_k)
  return [(doc, 1.0/ (1.0 + score)) for doc, score in results] # 거리기반인 similarity... 결과의 역수를 취해줌

query = "트랜스포머와 버트의 관계"
results = vector_search(query, vectorstore)

In [ ]:
relevant = {'d1', 'd2', 'd3'}
orig_order = [doc.metadata['id'] for doc, _ in results]

In [ ]:
ap = average_precision(orig_order, relevant)

In [ ]:
ap

0.9166666666666666

### ILS (Intra-List Similarity)

검색 결과 목록 내 문서들의 상호 유사도.

- ILS 낮음 → 결과가 다양 (사용자에게 다양한 정보 제공)
- ILS 높음 → 결과가 유사 (중복 위험)

mAP는 정확도 중심이므로 ILS로 다양성을 별도 평가.

In [ ]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k = 5):
  ids = doc_ids[:top_k]
  embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]

  if len(embs) < 2:
    return 0.0

  sims = []
  for i in range(len(embs)):
    for j in range(i+1, len(embs)):
      sims.append(cosine_sim(embs[i], embs[j]))

  return np.mean(sims)

In [ ]:
ils_score = intra_list_similarity(orig_order, doc_embeddings)

NameError: name 'doc_embeddings' is not defined